In [1]:
# ============================================================================
# ENHANCED TRAINING AND PREDICTION PIPELINE FOR TELECOM CHURN
# ============================================================================
import pandas as pd
import numpy as np
import re
from datetime import datetime
from pathlib import Path
import sys

# Add project root to path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, QuantileTransformer
from sklearn.metrics import classification_report, accuracy_score, make_scorer, roc_auc_score, f1_score, confusion_matrix
from sklearn.feature_selection import SelectFromModel, VarianceThreshold, mutual_info_classif
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

# Import custom feature engineering utilities
from utils.feature_engineering import (
    AdvancedImputer,
    IntelligentFeatureSelector,
    TelecomFeatureEngineer,
)


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================
def clean_feature_names(columns):
    """Clean feature names to remove special characters for LightGBM compatibility"""
    cleaned = []
    for col in columns:
        cleaned_name = re.sub(r'[^A-Za-z0-9_]', '_', str(col))
        cleaned_name = re.sub(r'_+', '_', cleaned_name)
        cleaned_name = cleaned_name.strip('_')
        cleaned.append(cleaned_name)
    return cleaned


def get_feature_columns(df):
    """Get the list of feature columns to use for modeling"""
    exclude_cols = {'Unnamed: 0', 'id', 'label', 'age_group', 'customer_segment'}
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    return feature_cols


def reduce_features_by_importance(X, y, model, threshold=0.001):
    """
    Reduce features based on feature importance
    """
    print(f"\nPerforming feature selection (threshold={threshold})...")
    selector = SelectFromModel(model, threshold=threshold, prefit=False)
    selector.fit(X, y)
    selected_features = X.columns[selector.get_support()].tolist()
    print(f"Selected {len(selected_features)} features out of {len(X.columns)}")
    return selected_features


# Define custom scoring function
def custom_score(y_true, y_pred, y_proba=None):
    """Calculate custom score: 0.7 * accuracy + 0.3 * f1"""
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    return 0.7 * acc + 0.3 * f1


custom_scorer = make_scorer(custom_score, needs_proba=False)


from sklearn.metrics import precision_recall_curve


def find_optimal_threshold(y_true, y_proba, metric='f1'):
    """Find optimal threshold for F1 score"""
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * (precision * recall) / (precision + recall)
    
    # Handle NaN values
    f1_scores = np.nan_to_num(f1_scores)
    
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx]
    optimal_f1 = f1_scores[optimal_idx]
    
    return optimal_threshold, optimal_f1


# ============================================================================
# LOAD AND PREPARE TRAINING DATA
# ============================================================================
print("=" * 80)
print("LOADING TRAINING DATA")
print("=" * 80)

df_train = pd.read_csv('./train.csv')
print(f"Training data shape: {df_train.shape}")
print(f"\nClass distribution:")
print(df_train['label'].value_counts())
print(f"Class ratio: {df_train['label'].value_counts()[0] / df_train['label'].value_counts()[1]:.2f}:1")

# Check for missing values
print(f"\nMissing values summary:")
missing_summary = df_train.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
if len(missing_summary) > 0:
    print(missing_summary.head(10))
else:
    print("No missing values found")

# ============================================================================
# ADVANCED FEATURE ENGINEERING WITH UTILITIES
# ============================================================================
print("\n" + "=" * 80)
print("CREATING ADVANCED FEATURES")
print("=" * 80)

# Step 1: Advanced imputation
print("Step 1: Performing advanced imputation...")
imputer = AdvancedImputer(strategy='auto')
df_train_imputed = imputer.fit_transform(df_train, target=df_train['label'])

# Step 2: Feature engineering
print("Step 2: Creating telecom-specific features...")
engineer = TelecomFeatureEngineer()
df_train_processed = engineer.create_all_features(
    df_train_imputed, 
    is_train=True, 
    target=df_train_imputed['label']
)

# Step 3: Get feature columns
exclude_cols = {'label', 'id', 'Unnamed: 0', 'age_group', 'customer_segment'}
feature_cols = [col for col in df_train_processed.columns if col not in exclude_cols]
print(f"Total features after engineering: {len(feature_cols)}")

# Prepare X and y
X = df_train_processed[feature_cols].copy()
y = df_train_processed['label']

# Handle remaining missing values and infinities
print("\nHandling missing values and infinities...")
X = X.replace([np.inf, -np.inf], 0)
X = X.fillna(0)

# Step 4: Intelligent feature selection
print("\nStep 4: Selecting most discriminative features...")
selector = IntelligentFeatureSelector(n_features=120)  # Keep top 120 features
selected_features = selector.select_features(X, y, method='ensemble')
X = X[selected_features]

print(f"Final feature count after selection: {X.shape[1]}")
print(f"\nTop 10 features by importance:")
for i, (feat, score) in enumerate(list(selector.feature_scores.items())[:10], 1):
    print(f"  {i}. {feat}: {score:.4f}")

# Clean feature names for compatibility
original_feature_cols = X.columns.tolist()
cleaned_feature_cols = clean_feature_names(original_feature_cols)
feature_name_mapping = dict(zip(cleaned_feature_cols, original_feature_cols))
X.columns = cleaned_feature_cols

print(f"\nFinal feature matrix shape: {X.shape}")
print(f"Feature statistics:")
print(f"  Numeric features: {X.select_dtypes(include=[np.number]).shape[1]}")
print(f"  Memory usage: {X.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# ============================================================================
# FEATURE SCALING (Optional - for some models)
# ============================================================================
print("\n" + "=" * 80)
print("FEATURE SCALING")
print("=" * 80)

# Create scaled version for models that benefit from it
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)
print("Features scaled using StandardScaler")

# ============================================================================
# TRAIN-VALIDATION SPLIT
# ============================================================================
print("\n" + "=" * 80)
print("SPLITTING DATA FOR VALIDATION")
print("=" * 80)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Training class distribution: {y_train.value_counts().to_dict()}")
print(f"Validation class distribution: {y_val.value_counts().to_dict()}")

LOADING TRAINING DATA
Training data shape: (59904, 88)

Class distribution:
label
0    44904
1    15000
Name: count, dtype: int64
Class ratio: 2.99:1

Missing values summary:
gm_use_dur               47451
gm_dayt_use_dur          47451
gm_ngt_use_dur           47451
shrt_vid_use_dur         47451
shrt_vid_dayt_use_dur    47451
shrt_vid_ngt_use_dur     47451
long_vid_use_dur         47451
long_vid_dayt_use_dur    47451
long_vid_ngt_use_dur     47451
anchor_use_dur           47451
dtype: int64

CREATING ADVANCED FEATURES
Step 1: Performing advanced imputation...
Step 2: Creating telecom-specific features...

ADVANCED FEATURE ENGINEERING
Creating usage pattern features...
Creating temporal features...
Creating behavioral segments...
Creating revenue features...
Creating engagement metrics...
Creating risk indicators...
Creating interaction features...
Creating statistical features...
Creating target-encoded features...
Creating polynomial features...
Creating aggregation features...
Appl

In [2]:

# ============================================================================
# ADVANCED MODEL TRAINING WITH CROSS-VALIDATION
# ============================================================================
print("\n" + "=" * 80)
print("TRAINING ADVANCED MODELS WITH CROSS-VALIDATION")
print("=" * 80)

# Calculate scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

# Setup cross-validation
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

# Store results
results = {}



TRAINING ADVANCED MODELS WITH CROSS-VALIDATION
Scale pos weight: 2.99


In [6]:

# ============================================================================
# MODEL 1: LIGHTGBM WITH HYPERPARAMETER TUNING
# ============================================================================
print("\n" + "-" * 80)
print("1. TRAINING LIGHTGBM")
print("-" * 80)

lgb_params_enhanced = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 1500,  # Increased
    'max_depth': 12,       # Slightly deeper
    'learning_rate': 0.02, # Lower learning rate
    'num_leaves': 60,      # More leaves
    'subsample': 0.85,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    'min_child_samples': 25,
    'min_child_weight': 0.001,
    'min_split_gain': 0.005,
    'reg_alpha': 0.4,
    'reg_lambda': 0.4,
    # 'class_weight': 'balanced',
    'random_state': 42,
    'feature_fraction_seed': 42,
    'bagging_seed': 42,
    'drop_seed': 42,
    'data_random_seed': 42,
    'extra_trees': True,      # Extra randomization
    'path_smooth': 0.1,       # Path smoothing
    'verbose': -1,
    'force_col_wise': True
}

lgb_model = lgb.LGBMClassifier(**lgb_params_enhanced)

# # Cross-validation
# print("Performing 5-fold cross-validation...")
# cv_scores_lgb = cross_val_score(lgb_model, X_train, y_train, cv=skf, 
#                                  scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_lgb}")
# print(f"Mean CV ROC-AUC: {cv_scores_lgb.mean():.4f} (+/- {cv_scores_lgb.std() * 2:.4f})")

# Train on full training set with early stopping
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

y_pred_lgb = lgb_model.predict(X_val)
y_proba_lgb = lgb_model.predict_proba(X_val)[:, 1]


lgb_acc = accuracy_score(y_val, y_pred_lgb)
lgb_auc = roc_auc_score(y_val, y_proba_lgb)
lgb_f1 = f1_score(y_val, y_pred_lgb)
custom_score_lgb = custom_score(y_val, y_pred_lgb, y_proba_lgb)

print(f"\nValidation Metrics:")
print(f"  Accuracy: {lgb_acc:.4f}")
print(f"  ROC-AUC: {lgb_auc:.4f}")
print(f"  F1-Score: {lgb_f1:.4f}")
print(f"  Custom Score: {custom_score_lgb:.4f}")

results['LightGBM'] = {
    'model': lgb_model,
    # 'cv_scores': cv_scores_lgb,
    'val_auc': lgb_auc,
    'val_f1': lgb_f1,
    'predictions': y_pred_lgb,
    'probabilities': y_proba_lgb,
    'custom_score': custom_score_lgb
}

# ============================================================================
# MODEL 1.1: OPTIMIZED FocalLossLGBM
# ============================================================================
print("\n" + "-" * 80)
print("1.1. TRAINING OPTIMIZED FocalLossLGBM")
print("-" * 80)

import numpy as np
import lightgbm as lgb
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import ParameterGrid

def improved_focal_loss_lgb(y_pred, y_true, alpha=0.75, gamma=1.5):
    """
    Improved Focal Loss for LightGBM with better numerical stability
    """
    y_true = y_true.get_label()
    
    # Apply sigmoid with numerical stability
    y_pred = np.clip(y_pred, -250, 250)  # Prevent overflow
    p = 1.0 / (1.0 + np.exp(-y_pred))
    p = np.clip(p, 1e-7, 1 - 1e-7)  # Prevent log(0)
    
    # Calculate focal loss
    alpha_t = np.where(y_true == 1, alpha, 1 - alpha)
    pt = np.where(y_true == 1, p, 1 - p)
    focal_weight = alpha_t * np.power(1 - pt, gamma)
    
    # Binary cross entropy loss
    bce = -np.where(y_true == 1, np.log(p), np.log(1 - p))
    
    # Focal loss
    focal_loss = focal_weight * bce
    
    # Gradients
    grad = np.where(y_true == 1,
                    alpha * (1 - p)**gamma * (gamma * p * np.log(p) + p - 1),
                    -(1 - alpha) * p**gamma * (gamma * (1 - p) * np.log(1 - p) + p))
    
    # Hessians (more stable calculation)
    hess = focal_weight * p * (1 - p) * (1 + gamma * np.log(pt))
    hess = np.maximum(hess, 1e-6)  # Ensure positive hessian
    
    return grad, hess

# Calculate optimal class weights
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))
print(f"Class weights: {class_weight_dict}")

# Calculate scale_pos_weight for imbalanced data
pos_count = np.sum(y_train == 1)
neg_count = np.sum(y_train == 0)
scale_pos_weight = neg_count / pos_count
print(f"Scale pos weight: {scale_pos_weight:.4f}")

# Enhanced parameters for focal loss
focal_params_enhanced = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 2000,
    'max_depth': 8,
    'learning_rate': 0.015,  # Lower learning rate
    'num_leaves': 45,
    'subsample': 0.9,
    'subsample_freq': 1,
    'colsample_bytree': 0.85,
    'min_child_samples': 20,
    'min_child_weight': 0.01,
    'min_split_gain': 0.01,
    'reg_alpha': 0.3,
    'reg_lambda': 0.5,
    'scale_pos_weight': scale_pos_weight,  # Better than class_weight for LGB
    'random_state': 42,
    'verbose': -1,
    'force_col_wise': True,
    'feature_fraction_seed': 42,
    'bagging_seed': 42,
    'extra_trees': True,
    'path_smooth': 0.05,
}

# Method 1: Enhanced class-weighted approach
print("Training enhanced class-weighted model...")
lgb_focal_enhanced = lgb.LGBMClassifier(**focal_params_enhanced)

# Use early stopping with validation set
lgb_focal_enhanced.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
)

y_pred_focal_enhanced = lgb_focal_enhanced.predict(X_val)
y_proba_focal_enhanced = lgb_focal_enhanced.predict_proba(X_val)[:, 1]

# Method 2: Custom focal loss with improved implementation
print("Training custom focal loss model...")
try:
    # Prepare datasets
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    # Custom focal loss parameters
    focal_custom_params = {
        'boosting_type': 'gbdt',
        'num_leaves': 45,
        'learning_rate': 0.02,
        'feature_fraction': 0.85,
        'bagging_fraction': 0.9,
        'bagging_freq': 5,
        'min_data_in_leaf': 20,
        'min_gain_to_split': 0.01,
        'lambda_l1': 0.3,
        'lambda_l2': 0.5,
        'verbose': -1,
        'random_state': 42,
        'max_depth': 8,
        'objective': improved_focal_loss_lgb,
    }
    
    # Train with custom focal loss
    focal_custom_model = lgb.train(
        focal_custom_params,
        train_data,
        valid_sets=[val_data],
        num_boost_round=2000,
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    
    # Predictions
    y_pred_focal_custom_raw = focal_custom_model.predict(X_val, num_iteration=focal_custom_model.best_iteration)
    y_proba_focal_custom = 1 / (1 + np.exp(-y_pred_focal_custom_raw))
    
    # Find optimal threshold
    from sklearn.metrics import precision_recall_curve
    precision, recall, thresholds = precision_recall_curve(y_val, y_proba_focal_custom)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
    optimal_threshold = thresholds[np.argmax(f1_scores)]
    
    y_pred_focal_custom = (y_proba_focal_custom > optimal_threshold).astype(int)
    
    print(f"Optimal threshold for custom focal loss: {optimal_threshold:.4f}")
    
    focal_custom_success = True
    
except Exception as e:
    print(f"Custom focal loss failed: {e}")
    focal_custom_success = False
    y_pred_focal_custom = y_pred_focal_enhanced
    y_proba_focal_custom = y_proba_focal_enhanced

# Method 3: Hybrid approach with threshold optimization
print("Optimizing prediction threshold...")
from sklearn.metrics import precision_recall_curve, f1_score

# Find optimal threshold for enhanced model
precision_enh, recall_enh, thresholds_enh = precision_recall_curve(y_val, y_proba_focal_enhanced)
f1_scores_enh = 2 * (precision_enh * recall_enh) / (precision_enh + recall_enh + 1e-8)
optimal_threshold_enh = thresholds_enh[np.argmax(f1_scores_enh)]

y_pred_focal_threshold = (y_proba_focal_enhanced > optimal_threshold_enh).astype(int)
print(f"Optimal threshold for enhanced model: {optimal_threshold_enh:.4f}")

# Evaluate all models
def evaluate_model(y_true, y_pred, y_proba, model_name):
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba)
    f1 = f1_score(y_true, y_pred)
    custom = custom_score(y_true, y_pred, y_proba)
    
    print(f"\n{model_name} Metrics:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  ROC-AUC: {auc:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  Custom Score: {custom:.4f}")
    
    return {'accuracy': acc, 'auc': auc, 'f1': f1, 'custom': custom}

# Evaluate all versions
enhanced_metrics = evaluate_model(y_val, y_pred_focal_enhanced, y_proba_focal_enhanced, 
                                "Enhanced Class-Weighted")
threshold_metrics = evaluate_model(y_val, y_pred_focal_threshold, y_proba_focal_enhanced, 
                                 "Threshold Optimized")

if focal_custom_success:
    custom_metrics = evaluate_model(y_val, y_pred_focal_custom, y_proba_focal_custom, 
                                   "Custom Focal Loss")
else:
    custom_metrics = enhanced_metrics

if enhanced_metrics['custom'] > threshold_metrics['custom']:
    results['LightGBM_enhanced'] = {
        'model': lgb_focal_enhanced,
        # 'cv_scores': cv_scores_lgb,
        'val_auc': enhanced_metrics['auc'],
        'val_f1': enhanced_metrics['f1'],
        'predictions': y_pred_focal_enhanced,
        'probabilities': y_proba_focal_enhanced,
        'custom_score': enhanced_metrics['custom']
    }
else:
    results['LightGBM_threshold'] = {
        'model': lgb_focal_enhanced,
        # 'cv_scores': cv_scores_lgb,
        'val_auc': threshold_metrics['auc'],
        'val_f1': threshold_metrics['f1'],
        'threshold': optimal_threshold_enh,
        'predictions': y_pred_focal_enhanced,
        'probabilities': y_proba_focal_enhanced,
        'custom_score': threshold_metrics['custom']
    }



--------------------------------------------------------------------------------
1. TRAINING LIGHTGBM
--------------------------------------------------------------------------------

Validation Metrics:
  Accuracy: 0.8061
  ROC-AUC: 0.8389
  F1-Score: 0.5520
  Custom Score: 0.7299

--------------------------------------------------------------------------------
1.1. TRAINING OPTIMIZED FocalLossLGBM
--------------------------------------------------------------------------------
Class weights: {np.int64(0): np.float64(0.6670247327604276), np.int64(1): np.float64(1.9967843137254901)}
Scale pos weight: 2.9936
Training enhanced class-weighted model...
Training custom focal loss model...
Custom focal loss failed: For early stopping, at least one dataset and eval metric is required for evaluation
Optimizing prediction threshold...
Optimal threshold for enhanced model: 0.5543

Enhanced Class-Weighted Metrics:
  Accuracy: 0.7614
  ROC-AUC: 0.8385
  F1-Score: 0.6154
  Custom Score: 0.7176

Th

In [9]:
# ============================================================================
# MODEL 2: XGBOOST WITH ADVANCED PARAMETERS
# ============================================================================
print("\n" + "-" * 80)
print("2. TRAINING XGBOOST")
print("-" * 80)

xgb_params_enhanced = {
    'n_estimators': 1500,
    'max_depth': 8,
    'learning_rate': 0.02,
    'subsample': 0.85,
    'colsample_bytree': 0.8,
    'colsample_bylevel': 0.8,
    'colsample_bynode': 0.8,
    'min_child_weight': 3,
    'gamma': 0.05,
    'reg_alpha': 0.4,
    'reg_lambda': 0.4,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'tree_method': 'hist',
    'grow_policy': 'depthwise',
    'max_bin': 512,          # Increased bins
    # 'sampling_method': 'gradient_based',
    'eval_metric': 'auc'
}

xgb_model = xgb.XGBClassifier(**xgb_params_enhanced)

# # Cross-validation
# print("Performing 5-fold cross-validation...")
# cv_scores_xgb = cross_val_score(xgb_model, X_train, y_train, cv=skf, 
#                                  scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_xgb}")
# print(f"Mean CV ROC-AUC: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std() * 2:.4f})")

# Train on full training set with early stopping
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    # early_stopping_rounds=50,
    verbose=False
)

y_pred_xgb = xgb_model.predict(X_val)
y_proba_xgb = xgb_model.predict_proba(X_val)[:, 1]

xgb_acc = accuracy_score(y_val, y_pred_xgb)
xgb_auc = roc_auc_score(y_val, y_proba_xgb)
xgb_f1 = f1_score(y_val, y_pred_xgb)
xgb_custom_score = custom_score(y_val, y_pred_xgb, y_proba_xgb)

print(f"\nValidation Metrics:")
print(f"  Accuracy: {xgb_acc:.4f}")
print(f"  ROC-AUC: {xgb_auc:.4f}")
print(f"  F1-Score: {xgb_f1:.4f}")
print(f"  Custom Score: {xgb_custom_score:.4f}")

results['XGBoost'] = {
    'model': xgb_model,
    # 'cv_scores': cv_scores_xgb,
    'val_auc': xgb_auc,
    'val_f1': xgb_f1,
    'predictions': y_pred_xgb,
    'probabilities': y_proba_xgb,
    'custom_score': xgb_custom_score
    
}


--------------------------------------------------------------------------------
2. TRAINING XGBOOST
--------------------------------------------------------------------------------

Validation Metrics:
  Accuracy: 0.7966
  ROC-AUC: 0.8429
  F1-Score: 0.6288
  Custom Score: 0.7462


In [16]:
# ============================================================================
# MODEL 2.1: XGBOOST WITH RAND SEARCH
# ============================================================================
print("\n" + "-" * 80)
print("MODEL 2.1: XGBOOST WITH RAND SEARCH")
print("-" * 80)

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from scipy.stats import randint, uniform

param_distributions = {
    'n_estimators': randint(1000, 3000),
    'max_depth': randint(4, 12),
    'learning_rate': uniform(0.01, 0.1),
    'subsample': uniform(0.7, 0.3),
    'colsample_bytree': uniform(0.7, 0.3),
    'colsample_bylevel': uniform(0.7, 0.3),
    'colsample_bynode': uniform(0.7, 0.3),
    'min_child_weight': randint(1, 10),
    'gamma': uniform(0, 0.2),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0, 1),
    'scale_pos_weight': randint(3, 4),
    'max_bin': randint(512, 513),          # Increased bins
    # 'sampling_method': 'gradient_based',
}

xgb_random = RandomizedSearchCV(
    xgb.XGBClassifier(random_state=42, tree_method='hist'),
    param_distributions=param_distributions,
    n_iter=50,
    cv=3,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1
)

# # Cross-validation
# print("Performing 5-fold cross-validation...")
# cv_scores_xgb = cross_val_score(xgb_model, X_train, y_train, cv=skf, 
#                                  scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_xgb}")
# print(f"Mean CV ROC-AUC: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std() * 2:.4f})")

# Train on full training set with early stopping
xgb_random.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    # early_stopping_rounds=50,
    verbose=False
)

xgb_random_best = xgb_random.best_estimator_

y_pred_xgb_rand = xgb_random_best.predict(X_val)
y_proba_xgb_rand = xgb_random_best.predict_proba(X_val)[:, 1]

xgb_rand_acc = accuracy_score(y_val, y_pred_xgb_rand)
xgb_rand_auc = roc_auc_score(y_val, y_proba_xgb_rand)
xgb_rand_f1 = f1_score(y_val, y_pred_xgb_rand)
xgb_rand_custom_score = custom_score(y_val, y_pred_xgb_rand, y_proba_xgb_rand)

print(f"\nValidation Metrics:")
print(f"  Accuracy: {xgb_rand_acc:.4f}")
print(f"  ROC-AUC: {xgb_rand_auc:.4f}")
print(f"  F1-Score: {xgb_rand_f1:.4f}")
print(f"  Custom Score: {xgb_rand_custom_score:.4f}")

results['XGBoost_rand'] = {
    'model': xgb_random_best,
    # 'cv_scores': cv_scores_xgb,
    'val_auc': xgb_rand_auc,
    'val_f1': xgb_rand_f1,
    'predictions': y_pred_xgb_rand,
    'probabilities': y_proba_xgb_rand,
    'custom_score': xgb_rand_custom_score
    
}


--------------------------------------------------------------------------------
MODEL 2.1: XGBOOST WITH RAND SEARCH
--------------------------------------------------------------------------------

Validation Metrics:
  Accuracy: 0.7823
  ROC-AUC: 0.8459
  F1-Score: 0.6304
  Custom Score: 0.7367


In [19]:
# ============================================================================
# MODEL 3: CATBOOST
# ============================================================================
print("\n" + "-" * 80)
print("3. TRAINING CATBOOST")
print("-" * 80)

catboost_params = {
    'iterations': 1000,
    'depth': 8,
    'learning_rate': 0.03,
    'l2_leaf_reg': 3,
    'subsample': 0.8,
    'colsample_bylevel': 0.8,
    'min_data_in_leaf': 20,
    'auto_class_weights': 'Balanced',
    'random_state': 42,
    'thread_count': -1,
    'verbose': False,
    'early_stopping_rounds': 50,
    'eval_metric': 'AUC',
    'border_count': 128
}

catboost_model = CatBoostClassifier(**catboost_params)

# # Cross-validation
# print("Performing 5-fold cross-validation...")
# cv_scores_cat = cross_val_score(catboost_model, X_train, y_train, cv=skf, 
#                                 scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_cat}")
# print(f"Mean CV ROC-AUC: {cv_scores_cat.mean():.4f} (+/- {cv_scores_cat.std() * 2:.4f})")

# Train on full training set
catboost_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    verbose=False
)

y_pred_cat = catboost_model.predict(X_val)
y_proba_cat = catboost_model.predict_proba(X_val)[:, 1]

cat_acc = accuracy_score(y_val, y_pred_cat)
cat_auc = roc_auc_score(y_val, y_proba_cat)
cat_f1 = f1_score(y_val, y_pred_cat)
cat_custom_score = custom_score(y_val, y_pred_cat, y_proba_cat)

print(f"\nValidation Metrics:")
print(f"  Accuracy: {cat_acc:.4f}")
print(f"  ROC-AUC: {cat_auc:.4f}")
print(f"  F1-Score: {cat_f1:.4f}")
print(f"  Custom Score: {cat_custom_score:.4f}")

results['CatBoost'] = {
    'model': catboost_model,
    # 'cv_scores': cv_scores_cat,
    'val_auc': cat_auc,
    'val_f1': cat_f1,
    'predictions': y_pred_cat,
    'probabilities': y_proba_cat,
    'custom_score': cat_custom_score
}



--------------------------------------------------------------------------------
3. TRAINING CATBOOST
--------------------------------------------------------------------------------

Validation Metrics:
  Accuracy: 0.7661
  ROC-AUC: 0.8427
  F1-Score: 0.6218
  Custom Score: 0.7228


In [24]:
# ============================================================================
# MODEL 3: ENHANCED CATBOOST
# ============================================================================
print("\n" + "-" * 80)
print("3. TRAINING ENHANCED CATBOOST")
print("-" * 80)

# Enhanced parameters with better tuning
catboost_params_enhanced = {
    'iterations': 2000,  # Increased iterations
    'depth': 10,  # Increased depth for more complexity
    'learning_rate': 0.01,  # Lower learning rate for better convergence
    'l2_leaf_reg': 5,  # Increased regularization
    # 'subsample': 0.85,  # Slightly higher subsample
    'colsample_bylevel': 0.9,  # Higher feature sampling
    'min_data_in_leaf': 15,  # Lower min data for finer splits
    'auto_class_weights': 'Balanced',
    'random_state': 42,
    'thread_count': -1,
    'verbose': 100,  # Show progress every 100 iterations
    'early_stopping_rounds': 100,  # More patience
    'eval_metric': 'AUC',
    'border_count': 254,  # Increased for better feature discretization
    'bagging_temperature': 1.0,  # Add some randomness
    # 'rsm': 0.9,  # Random subspace method
    'bootstrap_type': 'Bayesian',  # Better bootstrap method
    # 'od_type': 'Iter',  # Overfitting detection
    # 'od_wait': 50,  # Wait iterations for overfitting detection
}

# Initialize model
catboost_model_enhanced = CatBoostClassifier(**catboost_params_enhanced)

# Train with validation monitoring
print("\nTraining model with validation monitoring...")
catboost_model_enhanced.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True,  # Use best iteration based on validation
    verbose=100
)

# Get predictions
y_pred_cat = catboost_model_enhanced.predict(X_val)
y_proba_cat = catboost_model_enhanced.predict_proba(X_val)[:, 1]

# Calculate metrics
cat_acc = accuracy_score(y_val, y_pred_cat)
cat_auc = roc_auc_score(y_val, y_proba_cat)
cat_f1 = f1_score(y_val, y_pred_cat)
cat_custom_score = custom_score(y_val, y_pred_cat, y_proba_cat)

# Additional metrics
from sklearn.metrics import precision_score, recall_score, classification_report
cat_precision = precision_score(y_val, y_pred_cat)
cat_recall = recall_score(y_val, y_pred_cat)

print(f"\nValidation Metrics:")
print(f"  Accuracy: {cat_acc:.4f}")
print(f"  ROC-AUC: {cat_auc:.4f}")
print(f"  F1-Score: {cat_f1:.4f}")
print(f"  Precision: {cat_precision:.4f}")
print(f"  Recall: {cat_recall:.4f}")
print(f"  Custom Score: {cat_custom_score:.4f}")


# Threshold optimization
from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(y_val, y_proba_cat)
f1_scores = 2 * (precision * recall) / (precision + recall)
best_threshold_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_threshold_idx]

print(f"\nOptimal threshold for F1-score: {best_threshold:.4f}")

# Apply optimal threshold
y_pred_cat_optimized = (y_proba_cat >= best_threshold).astype(int)
cat_f1_optimized = f1_score(y_val, y_pred_cat_optimized)
cat_acc_optimized = accuracy_score(y_val, y_pred_cat_optimized)

print(f"Optimized F1-Score: {cat_f1_optimized:.4f}")
print(f"Optimized Accuracy: {cat_acc_optimized:.4f}")

# Store results
results['CatBoost_enhanced'] = {
    'model': catboost_model_enhanced,
    'val_auc': cat_auc,
    'val_f1': cat_f1,
    'val_f1_optimized': cat_f1_optimized,
    'predictions': y_pred_cat,
    'predictions_optimized': y_pred_cat_optimized,
    'probabilities': y_proba_cat,
    'custom_score': cat_custom_score,
    'best_threshold': best_threshold,
}



--------------------------------------------------------------------------------
3. TRAINING ENHANCED CATBOOST
--------------------------------------------------------------------------------

Training model with validation monitoring...
0:	test: 0.7844889	best: 0.7844889 (0)	total: 65.4ms	remaining: 2m 10s
100:	test: 0.8200949	best: 0.8200949 (100)	total: 3.9s	remaining: 1m 13s
200:	test: 0.8251683	best: 0.8251683 (200)	total: 7.78s	remaining: 1m 9s
300:	test: 0.8280220	best: 0.8280220 (300)	total: 11.6s	remaining: 1m 5s
400:	test: 0.8302491	best: 0.8302510 (399)	total: 15.5s	remaining: 1m 1s
500:	test: 0.8316991	best: 0.8316991 (500)	total: 19.4s	remaining: 58s
600:	test: 0.8327799	best: 0.8327799 (600)	total: 23.2s	remaining: 54.1s
700:	test: 0.8335528	best: 0.8335528 (700)	total: 27.1s	remaining: 50.3s
800:	test: 0.8341061	best: 0.8341061 (800)	total: 31.1s	remaining: 46.5s
900:	test: 0.8347344	best: 0.8347563 (897)	total: 34.9s	remaining: 42.6s
1000:	test: 0.8353910	best: 0.83539

In [25]:
# ============================================================================
# MODEL 4: RANDOM FOREST WITH OPTIMIZED PARAMETERS
# ============================================================================
print("\n" + "-" * 80)
print("4. TRAINING RANDOM FOREST")
print("-" * 80)

rf_params = {
    'n_estimators': 500,
    'max_depth': 15,
    'min_samples_split': 10,
    'min_samples_leaf': 5,
    'max_features': 'sqrt',
    'bootstrap': True,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1,
    'criterion': 'gini',
    'max_samples': 0.8
}

rf_model = RandomForestClassifier(**rf_params)

# # Cross-validation
# print("Performing 5-fold cross-validation...")
# cv_scores_rf = cross_val_score(rf_model, X_train, y_train, cv=skf, 
#                                scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_rf}")
# print(f"Mean CV ROC-AUC: {cv_scores_rf.mean():.4f} (+/- {cv_scores_rf.std() * 2:.4f})")

# Train on full training set
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_val)
y_proba_rf = rf_model.predict_proba(X_val)[:, 1]

rf_acc = accuracy_score(y_val, y_pred_rf)
rf_auc = roc_auc_score(y_val, y_proba_rf)
rf_f1 = f1_score(y_val, y_pred_rf)
rf_custom_score = custom_score(y_val, y_pred_rf, y_proba_rf)

print(f"\nValidation Metrics:")
print(f"  Accuracy: {rf_acc:.4f}")
print(f"  ROC-AUC: {rf_auc:.4f}")
print(f"  F1-Score: {rf_f1:.4f}")
print(f"  Custom Score: {rf_custom_score:.4f}")

results['RandomForest'] = {
    'model': rf_model,
    # 'cv_scores': cv_scores_rf,
    'val_auc': rf_auc,
    'val_f1': rf_f1,
    'predictions': y_pred_rf,
    'probabilities': y_proba_rf,
    'custom_score': rf_custom_score
}


--------------------------------------------------------------------------------
4. TRAINING RANDOM FOREST
--------------------------------------------------------------------------------

Validation Metrics:
  Accuracy: 0.7829
  ROC-AUC: 0.8347
  F1-Score: 0.6199
  Custom Score: 0.7340


In [27]:
# ============================================================================
# COMPREHENSIVE RESULTS ANALYSIS FROM ALL MODELS
# ============================================================================
print("\n" + "=" * 80)
print("COMPREHENSIVE MODEL PERFORMANCE ANALYSIS")
print("=" * 80)

# Create a detailed results table
print("\n" + "-" * 100)
print(f"{'Model Name':<30} | {'Accuracy':>10} | {'ROC-AUC':>10} | {'F1-Score':>10} | {'Custom Score':>13}")
print("-" * 100)

for model_name, model_info in sorted(results.items(), key=lambda x: x[1].get('custom_score', 0), reverse=True):
    if 'model' in model_info or 'probabilities' in model_info:
        # Calculate metrics if not already present
        if 'predictions' in model_info:
            acc = accuracy_score(y_val, model_info['predictions'])
        else:
            acc = 0.0
        
        auc = model_info.get('val_auc', 0.0)
        f1 = model_info.get('val_f1', 0.0)
        custom = model_info.get('custom_score', 0.0)
        
        print(f"{model_name:<30} | {acc:>10.4f} | {auc:>10.4f} | {f1:>10.4f} | {custom:>13.4f}")

print("-" * 100)

# ============================================================================
# DETAILED PERFORMANCE BREAKDOWN BY CATEGORY
# ============================================================================
print("\n" + "=" * 80)
print("PERFORMANCE BREAKDOWN BY MODEL CATEGORY")
print("=" * 80)

# Categorize models
base_models = ['LightGBM', 'LightGBM_enhanced', 'LightGBM_threshold', 
               'XGBoost', 'XGBoost_rand', 'CatBoost', 'CatBoost_enhanced', 'RandomForest']
ensemble_models = ['WeightedEnsemble', 'F1OptimizedEnsemble', 'VotingClassifier', 'stackingClassifier']

def print_category_stats(category_name, model_names):
    category_results = {k: v for k, v in results.items() if k in model_names}
    if not category_results:
        return
    
    print(f"\n{category_name}:")
    print("-" * 80)
    
    aucs = [r['val_auc'] for r in category_results.values() if 'val_auc' in r]
    f1s = [r['val_f1'] for r in category_results.values() if 'val_f1' in r]
    customs = [r['custom_score'] for r in category_results.values() if 'custom_score' in r]
    
    if aucs:
        print(f"  ROC-AUC:      Mean={np.mean(aucs):.4f}, Best={np.max(aucs):.4f}, Worst={np.min(aucs):.4f}")
    if f1s:
        print(f"  F1-Score:     Mean={np.mean(f1s):.4f}, Best={np.max(f1s):.4f}, Worst={np.min(f1s):.4f}")
    if customs:
        print(f"  Custom Score: Mean={np.mean(customs):.4f}, Best={np.max(customs):.4f}, Worst={np.min(customs):.4f}")
    
    # Show which model performed best
    if customs:
        best_model = max(category_results.items(), key=lambda x: x[1].get('custom_score', 0))
        print(f"  Best Model: {best_model[0]} (Custom Score: {best_model[1]['custom_score']:.4f})")

print_category_stats("BASE MODELS", base_models)
print_category_stats("ENSEMBLE MODELS", ensemble_models)

# ============================================================================
# CONFUSION MATRIX ANALYSIS FOR TOP 3 MODELS
# ============================================================================
print("\n" + "=" * 80)
print("CONFUSION MATRIX ANALYSIS - TOP 3 MODELS")
print("=" * 80)

# Get top 3 models by custom score
top_models = sorted(results.items(), key=lambda x: x[1].get('custom_score', 0), reverse=True)[:3]

for model_name, model_info in top_models:
    if 'predictions' in model_info:
        print(f"\n{model_name}:")
        print("-" * 60)
        
        cm = confusion_matrix(y_val, model_info['predictions'])
        tn, fp, fn, tp = cm.ravel()
        
        print(f"  Confusion Matrix:")
        print(f"    True Negatives:  {tn:5d}  |  False Positives: {fp:5d}")
        print(f"    False Negatives: {fn:5d}  |  True Positives:  {tp:5d}")
        
        # Calculate additional metrics
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0
        
        print(f"\n  Detailed Metrics:")
        print(f"    Sensitivity (Recall): {sensitivity:.4f}")
        print(f"    Specificity:          {specificity:.4f}")
        print(f"    Precision (PPV):      {precision:.4f}")
        print(f"    Negative Pred Value:  {npv:.4f}")
        print(f"    Accuracy:             {(tp + tn) / (tp + tn + fp + fn):.4f}")

# ============================================================================
# PERFORMANCE VISUALIZATION DATA
# ============================================================================
print("\n" + "=" * 80)
print("MODEL RANKING BY DIFFERENT METRICS")
print("=" * 80)

metrics_dict = {
    'ROC-AUC': {},
    'F1-Score': {},
    'Custom Score': {}
}

for model_name, model_info in results.items():
    if 'val_auc' in model_info:
        metrics_dict['ROC-AUC'][model_name] = model_info['val_auc']
    if 'val_f1' in model_info:
        metrics_dict['F1-Score'][model_name] = model_info['val_f1']
    if 'custom_score' in model_info:
        metrics_dict['Custom Score'][model_name] = model_info['custom_score']

for metric_name, metric_values in metrics_dict.items():
    if metric_values:
        print(f"\n{metric_name} Rankings:")
        print("-" * 60)
        sorted_models = sorted(metric_values.items(), key=lambda x: x[1], reverse=True)
        for rank, (model_name, score) in enumerate(sorted_models[:5], 1):
            print(f"  {rank}. {model_name:<25} {score:.4f}")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("EXECUTIVE SUMMARY")
print("=" * 80)

all_customs = [r['custom_score'] for r in results.values() if 'custom_score' in r]
all_aucs = [r['val_auc'] for r in results.values() if 'val_auc' in r]
all_f1s = [r['val_f1'] for r in results.values() if 'val_f1' in r]

print(f"\nTotal Models Evaluated: {len(results)}")
print(f"\nOverall Performance Ranges:")
print(f"  ROC-AUC:      {min(all_aucs):.4f} - {max(all_aucs):.4f}")
print(f"  F1-Score:     {min(all_f1s):.4f} - {max(all_f1s):.4f}")
print(f"  Custom Score: {min(all_customs):.4f} - {max(all_customs):.4f}")

best_model_name = max(results.items(), key=lambda x: x[1].get('custom_score', 0))[0]
best_model_info = results[best_model_name]

print(f"\n🏆 BEST OVERALL MODEL: {best_model_name}")
print(f"   ROC-AUC:      {best_model_info['val_auc']:.4f}")
print(f"   F1-Score:     {best_model_info['val_f1']:.4f}")
print(f"   Custom Score: {best_model_info['custom_score']:.4f}")

# Show performance improvement
baseline_model = 'LightGBM' if 'LightGBM' in results else list(results.keys())[0]
if baseline_model != best_model_name and baseline_model in results:
    baseline_score = results[baseline_model]['custom_score']
    best_score = best_model_info['custom_score']
    improvement = ((best_score - baseline_score) / baseline_score) * 100
    
    print(f"\n📈 Improvement over {baseline_model}:")
    print(f"   Custom Score: {baseline_score:.4f} → {best_score:.4f}")
    print(f"   Improvement:  {improvement:+.2f}%")

print("\n" + "=" * 80)


COMPREHENSIVE MODEL PERFORMANCE ANALYSIS

----------------------------------------------------------------------------------------------------
Model Name                     |   Accuracy |    ROC-AUC |   F1-Score |  Custom Score
----------------------------------------------------------------------------------------------------
XGBoost                        |     0.7966 |     0.8429 |     0.6288 |        0.7462
XGBoost_rand                   |     0.7823 |     0.8459 |     0.6304 |        0.7367
LightGBM_threshold             |     0.7614 |     0.8385 |     0.6249 |        0.7364
RandomForest                   |     0.7829 |     0.8347 |     0.6199 |        0.7340
LightGBM                       |     0.8061 |     0.8389 |     0.5520 |        0.7299
CatBoost_enhanced              |     0.7735 |     0.8394 |     0.6215 |        0.7279
CatBoost                       |     0.7661 |     0.8427 |     0.6218 |        0.7228
FocalLossLGBM                  |     0.7601 |     0.8371 |     0.61

In [28]:
# ============================================================================
# ENSEMBLE METHODS
# ============================================================================
print("\n" + "=" * 80)
print("CREATING ENSEMBLE MODELS")
print("=" * 80)

# ============================================================================
# 1. THRESHOLD-AWARE WEIGHTED AVERAGE ENSEMBLE
# ============================================================================
print("\n" + "-" * 80)
print("1. THRESHOLD-AWARE WEIGHTED AVERAGE ENSEMBLE")
print("-" * 80)

# Use custom scores as weights and apply model-specific thresholds
weights_dict = {
    'LightGBM': results['LightGBM']['custom_score'],
    'LightGBM_threshold': results['LightGBM_threshold']['custom_score'], 
    'XGBoost': results['XGBoost']['custom_score'],
    'CatBoost_enhanced': results['CatBoost_enhanced']['custom_score'],
    'RandomForest': results['RandomForest']['custom_score']
}

# Normalize weights
total_weight = sum(weights_dict.values())
normalized_weights = {k: v/total_weight for k, v in weights_dict.items()}

print("Ensemble weights:")
for model, weight in normalized_weights.items():
    print(f"  {model}: {weight:.3f}")

# Get threshold-adjusted predictions and probabilities
def get_threshold_adjusted_predictions(model_name, model_results):
    """Get predictions using model-specific thresholds where available"""
    probabilities = model_results['probabilities']
    
    if model_name == 'LightGBM_threshold' and hasattr(model_results.get('model'), 'threshold'):
        threshold = model_results['model'].threshold
        predictions = (probabilities >= threshold).astype(int)
        print(f"  Using threshold {threshold:.3f} for {model_name}")
    elif model_name == 'CatBoost_enhanced' and hasattr(model_results.get('model'), 'best_threshold'):
        threshold = model_results['model'].best_threshold
        predictions = (probabilities >= threshold).astype(int)
        print(f"  Using threshold {threshold:.3f} for {model_name}")
    else:
        predictions = (probabilities >= 0.5).astype(int)
        print(f"  Using default threshold 0.5 for {model_name}")
    
    return predictions, probabilities

# Create weighted ensemble using threshold-adjusted predictions
ensemble_proba = np.zeros_like(y_val, dtype=float)
ensemble_pred_weighted = np.zeros_like(y_val, dtype=float)

print("\nApplying model-specific thresholds:")
for model_name, weight in normalized_weights.items():
    if model_name in results:
        adjusted_pred, proba = get_threshold_adjusted_predictions(model_name, results[model_name])
        ensemble_proba += weight * proba
        ensemble_pred_weighted += weight * adjusted_pred

ensemble_pred = (ensemble_pred_weighted >= 0.5).astype(int)

# Calculate metrics
ensemble_acc = accuracy_score(y_val, ensemble_pred)
ensemble_auc = roc_auc_score(y_val, ensemble_proba)
ensemble_f1 = f1_score(y_val, ensemble_pred)
ensemble_custom_score = custom_score(y_val, ensemble_pred, ensemble_proba)

print(f"\nThreshold-Aware Weighted Ensemble Metrics:")
print(f"  Accuracy: {ensemble_acc:.4f}")
print(f"  ROC-AUC: {ensemble_auc:.4f}")
print(f"  F1-Score: {ensemble_f1:.4f}")
print(f"  Custom Score: {ensemble_custom_score:.4f}")

results['ThresholdAwareEnsemble'] = {
    'val_auc': ensemble_auc,
    'val_f1': ensemble_f1,
    'predictions': ensemble_pred,
    'probabilities': ensemble_proba,
    'weights': normalized_weights,
    'custom_score': ensemble_custom_score
}

# ============================================================================
# 2. ADAPTIVE THRESHOLD ENSEMBLE
# ============================================================================
print("\n" + "-" * 80)
print("2. ADAPTIVE THRESHOLD ENSEMBLE")
print("-" * 80)

class AdaptiveThresholdEnsemble:
    def __init__(self):
        self.models = {}
        self.weights = {}
        self.thresholds = {}
    
    def add_model(self, model_name, model_results):
        """Add model with its specific threshold and weight"""
        if 'model' not in model_results:
            return
            
        model = model_results['model']
        weight = model_results['custom_score']
        
        # Extract model-specific threshold
        if model_name == 'LightGBM_threshold' and hasattr(model, 'threshold'):
            threshold = model.threshold
        elif model_name == 'CatBoost_enhanced' and hasattr(model, 'best_threshold'):
            threshold = model.best_threshold
        else:
            # Find optimal threshold for other models
            y_proba = model_results['probabilities']
            threshold, _ = find_optimal_threshold(y_val, y_proba)
        
        self.models[model_name] = model
        self.weights[model_name] = weight
        self.thresholds[model_name] = threshold
        
        print(f"  Added {model_name} with threshold {threshold:.3f}, weight {weight:.3f}")
    
    def predict_proba(self, X):
        """Predict probabilities using weighted average"""
        total_weight = sum(self.weights.values())
        ensemble_proba = np.zeros(X.shape[0])
        
        for model_name, model in self.models.items():
            if hasattr(model, 'predict_proba'):
                proba = model.predict_proba(X)[:, 1]
            else:
                proba = model.predict(X)
            
            weight = self.weights[model_name] / total_weight
            ensemble_proba += weight * proba
        
        return ensemble_proba
    
    def predict(self, X):
        """Predict using adaptive thresholding strategy"""
        # Get individual model predictions with their thresholds
        model_predictions = {}
        model_confidences = {}
        
        for model_name, model in self.models.items():
            if hasattr(model, 'predict_proba'):
                proba = model.predict_proba(X)[:, 1]
            else:
                proba = model.predict(X)
            
            threshold = self.thresholds[model_name]
            pred = (proba >= threshold).astype(int)
            confidence = np.abs(proba - threshold)  # Distance from threshold
            
            model_predictions[model_name] = pred
            model_confidences[model_name] = confidence
        
        # Adaptive voting based on confidence
        final_predictions = np.zeros(X.shape[0])
        
        for i in range(X.shape[0]):
            # Get confidence-weighted vote for each sample
            weighted_vote = 0
            total_confidence = 0
            
            for model_name in self.models.keys():
                confidence = model_confidences[model_name][i]
                weight = self.weights[model_name]
                vote = model_predictions[model_name][i]
                
                weighted_vote += confidence * weight * vote
                total_confidence += confidence * weight
            
            if total_confidence > 0:
                final_predictions[i] = 1 if weighted_vote / total_confidence > 0.5 else 0
            else:
                # Fallback to simple weighted vote
                simple_vote = sum(self.weights[mn] * model_predictions[mn][i] 
                                for mn in self.models.keys())
                final_predictions[i] = 1 if simple_vote > sum(self.weights.values()) / 2 else 0
        
        return final_predictions.astype(int)

# Create and train adaptive ensemble
adaptive_ensemble = AdaptiveThresholdEnsemble()

print("Building Adaptive Threshold Ensemble:")
key_models = ['LightGBM', 'LightGBM_threshold', 'XGBoost', 'CatBoost_enhanced', 'RandomForest']
for model_name in key_models:
    if model_name in results:
        adaptive_ensemble.add_model(model_name, results[model_name])

# Make predictions
adaptive_proba = adaptive_ensemble.predict_proba(X_val)
adaptive_pred = adaptive_ensemble.predict(X_val)

# Calculate metrics
adaptive_acc = accuracy_score(y_val, adaptive_pred)
adaptive_auc = roc_auc_score(y_val, adaptive_proba)
adaptive_f1 = f1_score(y_val, adaptive_pred)
adaptive_custom_score = custom_score(y_val, adaptive_pred, adaptive_proba)

print(f"\nAdaptive Threshold Ensemble Metrics:")
print(f"  Accuracy: {adaptive_acc:.4f}")
print(f"  ROC-AUC: {adaptive_auc:.4f}")
print(f"  F1-Score: {adaptive_f1:.4f}")
print(f"  Custom Score: {adaptive_custom_score:.4f}")

results['AdaptiveEnsemble'] = {
    'val_auc': adaptive_auc,
    'val_f1': adaptive_f1,
    'predictions': adaptive_pred,
    'probabilities': adaptive_proba,
    'custom_score': adaptive_custom_score,
    'model': adaptive_ensemble
}

# ============================================================================
# 3. STACKED ENSEMBLE WITH THRESHOLD OPTIMIZATION
# ============================================================================
print("\n" + "-" * 80)
print("3. STACKED ENSEMBLE WITH THRESHOLD OPTIMIZATION")
print("-" * 80)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

# Prepare stacking features using threshold-optimized predictions
stacking_features = []
feature_names = []

for model_name in ['LightGBM', 'LightGBM_threshold', 'XGBoost', 'CatBoost_enhanced', 'RandomForest']:
    if model_name in results:
        # Use probabilities as features
        probabilities = results[model_name]['probabilities']
        stacking_features.append(probabilities)
        feature_names.append(f"{model_name}_proba")
        
        # Use threshold-adjusted predictions as features
        if model_name == 'LightGBM_threshold' and hasattr(results[model_name].get('model'), 'threshold'):
            threshold = results[model_name]['model'].threshold
            threshold_pred = (probabilities >= threshold).astype(int)
        elif model_name == 'CatBoost_enhanced' and hasattr(results[model_name].get('model'), 'best_threshold'):
            threshold = results[model_name]['model'].best_threshold
            threshold_pred = (probabilities >= threshold).astype(int)
        else:
            threshold_pred = (probabilities >= 0.5).astype(int)
        
        stacking_features.append(threshold_pred)
        feature_names.append(f"{model_name}_pred")

X_stack = np.column_stack(stacking_features)

print(f"Stacking features shape: {X_stack.shape}")
print(f"Feature names: {feature_names}")

# Train meta-learner
meta_learner = LogisticRegression(random_state=42, max_iter=1000)
meta_learner.fit(X_stack, y_val)

# Make predictions
stack_proba = meta_learner.predict_proba(X_stack)[:, 1]
stack_pred = meta_learner.predict(X_stack)

# Calculate metrics
stack_acc = accuracy_score(y_val, stack_pred)
stack_auc = roc_auc_score(y_val, stack_proba)
stack_f1 = f1_score(y_val, stack_pred)
stack_custom_score = custom_score(y_val, stack_pred, stack_proba)

print(f"\nStacked Ensemble Metrics:")
print(f"  Accuracy: {stack_acc:.4f}")
print(f"  ROC-AUC: {stack_auc:.4f}")
print(f"  F1-Score: {stack_f1:.4f}")
print(f"  Custom Score: {stack_custom_score:.4f}")

# Show feature importance
feature_importance = abs(meta_learner.coef_[0])
print(f"\nMeta-learner Feature Importance:")
for name, importance in zip(feature_names, feature_importance):
    print(f"  {name}: {importance:.3f}")

results['StackedEnsemble'] = {
    'val_auc': stack_auc,
    'val_f1': stack_f1,
    'predictions': stack_pred,
    'probabilities': stack_proba,
    'custom_score': stack_custom_score,
    'meta_learner': meta_learner,
    'feature_names': feature_names
}

# ============================================================================
# ENSEMBLE COMPARISON
# ============================================================================
print("\n" + "=" * 80)
print("ENSEMBLE PERFORMANCE COMPARISON")
print("=" * 80)

ensemble_results = {
    'ThresholdAware': results['ThresholdAwareEnsemble'],
    'Adaptive': results['AdaptiveEnsemble'],
    'Stacked': results['StackedEnsemble']
}

print(f"{'Ensemble Method':<20} | {'Accuracy':>10} | {'ROC-AUC':>10} | {'F1-Score':>10} | {'Custom Score':>12}")
print("-" * 80)

for name, res in ensemble_results.items():
    acc = accuracy_score(y_val, res['predictions'])
    print(f"{name:<20} | {acc:>10.4f} | {res['val_auc']:>10.4f} | {res['val_f1']:>10.4f} | {res['custom_score']:>12.4f}")

# Find best ensemble
best_ensemble = max(ensemble_results.items(), key=lambda x: x[1]['custom_score'])
print(f"\nBest Ensemble: {best_ensemble[0]} (Custom Score: {best_ensemble[1]['custom_score']:.4f})")


CREATING ENSEMBLE MODELS

--------------------------------------------------------------------------------
1. THRESHOLD-AWARE WEIGHTED AVERAGE ENSEMBLE
--------------------------------------------------------------------------------
Ensemble weights:
  LightGBM: 0.199
  LightGBM_threshold: 0.200
  XGBoost: 0.203
  CatBoost_enhanced: 0.198
  RandomForest: 0.200

Applying model-specific thresholds:
  Using default threshold 0.5 for LightGBM
  Using default threshold 0.5 for LightGBM_threshold
  Using default threshold 0.5 for XGBoost
  Using default threshold 0.5 for CatBoost_enhanced
  Using default threshold 0.5 for RandomForest

Threshold-Aware Weighted Ensemble Metrics:
  Accuracy: 0.7877
  ROC-AUC: 0.8430
  F1-Score: 0.6275
  Custom Score: 0.7396

--------------------------------------------------------------------------------
2. ADAPTIVE THRESHOLD ENSEMBLE
--------------------------------------------------------------------------------
Building Adaptive Threshold Ensemble:
  Added

In [38]:
# ============================================================================
# VOTING CLASSIFIER WITH THRESHOLD OPTIMIZATION
# ============================================================================
print("\n" + "-" * 80)
print("2. VOTING CLASSIFIER WITH THRESHOLD OPTIMIZATION")
print("-" * 80)

# Define weights based on custom scores
weights_voting = np.array([
    results['XGBoost']['custom_score'],
    results['XGBoost_rand']['custom_score'],
    results['LightGBM_threshold']['custom_score'],
    results['RandomForest']['custom_score'],
    results['LightGBM']['custom_score'],
    results['CatBoost_enhanced']['custom_score']
])

# Normalize weights
weights_voting = weights_voting / weights_voting.sum()

print("Voting weights:")
model_names = ['XGBoost', 'XGBoost_rand', 'LightGBM_threshold', 'RandomForest', 'LightGBM', 'CatBoost_enhanced']
for name, weight in zip(model_names, weights_voting):
    print(f"  {name}: {weight:.3f}")

# Create voting estimators list
voting_estimators = [
    ('xgb', results['XGBoost']['model']),
    ('xgb_rand', results['XGBoost_rand']['model']),
    ('lgb_thresh', results['LightGBM_threshold']['model']),
    ('rf', results['RandomForest']['model']),
    ('lgb', results['LightGBM']['model']),
    ('cat_enh', results['CatBoost_enhanced']['model'])
]

# Standard Voting Classifier
voting_clf = VotingClassifier(
    estimators=voting_estimators,
    voting='soft',
    weights=weights_voting,
    n_jobs=-1
)

# Train voting classifier
print("\nTraining standard voting classifier...")
voting_clf.fit(X_train, y_train)
y_pred_voting = voting_clf.predict(X_val)
y_proba_voting = voting_clf.predict_proba(X_val)[:, 1]

voting_acc = accuracy_score(y_val, y_pred_voting)
voting_auc = roc_auc_score(y_val, y_proba_voting)
voting_f1 = f1_score(y_val, y_pred_voting)
voting_custom_score = custom_score(y_val, y_pred_voting, y_proba_voting)

print(f"\nStandard Voting Classifier Metrics:")
print(f"  Accuracy: {voting_acc:.4f}")
print(f"  ROC-AUC: {voting_auc:.4f}")
print(f"  F1-Score: {voting_f1:.4f}")
print(f"  Custom Score: {voting_custom_score:.4f}")

results['VotingClassifier'] = {
    'model': voting_clf,
    'val_auc': voting_auc,
    'val_f1': voting_f1,
    'predictions': y_pred_voting,
    'probabilities': y_proba_voting,
    'custom_score': voting_custom_score
}

# ============================================================================
# THRESHOLD-AWARE VOTING CLASSIFIER
# ============================================================================
print("\n" + "-" * 80)
print("2b. THRESHOLD-AWARE VOTING CLASSIFIER")
print("-" * 80)

class ThresholdAwareVotingClassifier:
    def __init__(self, estimators, weights=None):
        self.estimators = estimators  # List of (name, model) tuples
        self.weights = weights if weights is not None else np.ones(len(estimators)) / len(estimators)
        self.thresholds = {}
        self.fitted_estimators = {}
    
    def fit(self, X, y, X_val=None, y_val=None):
        """Fit all estimators and determine optimal thresholds"""
        print("Fitting estimators and determining thresholds...")
        
        for i, (name, estimator) in enumerate(self.estimators):
            print(f"  Fitting {name}...")
            
            # Fit the estimator if not already fitted
            if hasattr(estimator, 'fit') and not hasattr(estimator, 'classes_'):
                estimator.fit(X, y)
            
            self.fitted_estimators[name] = estimator
            
            # Determine threshold for this estimator
            if X_val is not None and y_val is not None:
                # Use validation set to find optimal threshold
                if hasattr(estimator, 'predict_proba'):
                    y_proba = estimator.predict_proba(X_val)[:, 1]
                else:
                    y_proba = estimator.predict(X_val)
                
                # Check if model already has optimized threshold
                if name == 'lgb_thresh' and hasattr(estimator, 'threshold'):
                    threshold = estimator.threshold
                    print(f"    Using existing threshold {threshold:.3f} for {name}")
                elif name == 'cat_enh' and hasattr(estimator, 'best_threshold'):
                    threshold = estimator.best_threshold
                    print(f"    Using existing threshold {threshold:.3f} for {name}")
                else:
                    # Find optimal threshold
                    threshold, best_f1 = find_optimal_threshold(y_val, y_proba)
                    print(f"    Optimal threshold {threshold:.3f} (F1: {best_f1:.3f}) for {name}")
                
                self.thresholds[name] = threshold
            else:
                # Default threshold
                self.thresholds[name] = 0.5
                print(f"    Using default threshold 0.5 for {name}")
        
        return self
    
    def predict_proba(self, X):
        """Get weighted average of probabilities"""
        proba_sum = np.zeros(X.shape[0])
        total_weight = 0
        
        for i, (name, estimator) in enumerate(self.estimators):
            if hasattr(estimator, 'predict_proba'):
                proba = estimator.predict_proba(X)[:, 1]
            else:
                proba = estimator.predict(X)
            
            weight = self.weights[i]
            proba_sum += weight * proba
            total_weight += weight
        
        return proba_sum / total_weight
    
    def predict(self, X):
        """Make predictions using threshold-aware voting"""
        # Method 1: Weighted probability with global threshold
        avg_proba = self.predict_proba(X)
        global_threshold_pred = (avg_proba >= 0.5).astype(int)
        
        # Method 2: Individual threshold voting
        votes = np.zeros(X.shape[0])
        total_weight = 0
        
        for i, (name, estimator) in enumerate(self.estimators):
            if hasattr(estimator, 'predict_proba'):
                proba = estimator.predict_proba(X)[:, 1]
            else:
                proba = estimator.predict(X)
            
            # Apply model-specific threshold
            threshold = self.thresholds.get(name, 0.5)
            model_pred = (proba >= threshold).astype(int)
            
            weight = self.weights[i]
            votes += weight * model_pred
            total_weight += weight
        
        threshold_voting_pred = (votes / total_weight >= 0.5).astype(int)
        
        # Combine both methods
        final_pred = np.where(
            np.abs(avg_proba - 0.5) > 0.1,  # High confidence
            global_threshold_pred,
            threshold_voting_pred  # Low confidence, use threshold voting
        )
        
        return final_pred

# Create threshold-aware voting classifier
threshold_voting_clf = ThresholdAwareVotingClassifier(
    estimators=voting_estimators,
    weights=weights_voting
)

# Fit with validation data for threshold optimization
threshold_voting_clf.fit(X_train, y_train, X_val, y_val)

# Make predictions
y_pred_thresh_voting = threshold_voting_clf.predict(X_val)
y_proba_thresh_voting = threshold_voting_clf.predict_proba(X_val)

# Calculate metrics
thresh_voting_acc = accuracy_score(y_val, y_pred_thresh_voting)
thresh_voting_auc = roc_auc_score(y_val, y_proba_thresh_voting)
thresh_voting_f1 = f1_score(y_val, y_pred_thresh_voting)
thresh_voting_custom_score = custom_score(y_val, y_pred_thresh_voting, y_proba_thresh_voting)

print(f"\nThreshold-Aware Voting Classifier Metrics:")
print(f"  Accuracy: {thresh_voting_acc:.4f}")
print(f"  ROC-AUC: {thresh_voting_auc:.4f}")
print(f"  F1-Score: {thresh_voting_f1:.4f}")
print(f"  Custom Score: {thresh_voting_custom_score:.4f}")

print(f"\nModel-specific thresholds used:")
for name, threshold in threshold_voting_clf.thresholds.items():
    print(f"  {name}: {threshold:.3f}")

results['ThresholdVotingClassifier'] = {
    'model': threshold_voting_clf,
    'val_auc': thresh_voting_auc,
    'val_f1': thresh_voting_f1,
    'predictions': y_pred_thresh_voting,
    'probabilities': y_proba_thresh_voting,
    'custom_score': thresh_voting_custom_score
}

# ============================================================================
# OPTIMIZED WEIGHTED VOTING
# ============================================================================
print("\n" + "-" * 80)
print("2c. OPTIMIZED WEIGHTED VOTING")
print("-" * 80)

# Find optimal weights using validation performance
from scipy.optimize import minimize

def voting_objective(weights, models_proba, y_true):
    """Objective function for weight optimization"""
    weights = weights / weights.sum()  # Normalize
    ensemble_proba = np.average(models_proba, axis=0, weights=weights)
    # Optimize for custom score
    ensemble_pred = (ensemble_proba >= 0.5).astype(int)
    score = -custom_score(y_true, ensemble_pred, ensemble_proba)  # Negative for minimization
    return score

# Get probabilities from all models
models_proba = np.array([
    results['XGBoost']['probabilities'],
    results['XGBoost_rand']['probabilities'],
    results['LightGBM_threshold']['probabilities'],
    results['RandomForest']['probabilities'],
    results['LightGBM']['probabilities'],
    results['CatBoost_enhanced']['probabilities']
])

# Initial weights (current custom scores)
initial_weights = weights_voting

# Optimize weights
print("Optimizing ensemble weights...")
result = minimize(
    voting_objective,
    initial_weights,
    args=(models_proba, y_val),
    method='SLSQP',
    bounds=[(0.01, 1.0)] * len(initial_weights),
    constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1}
)

optimized_weights = result.x
print("Optimization result:", "Success" if result.success else "Failed")
print("Original weights:", initial_weights)
print("Optimized weights:", optimized_weights)

# Create ensemble with optimized weights
optimized_ensemble_proba = np.average(models_proba, axis=0, weights=optimized_weights)
optimized_ensemble_pred = (optimized_ensemble_proba >= 0.5).astype(int)

# Calculate metrics
opt_voting_acc = accuracy_score(y_val, optimized_ensemble_pred)
opt_voting_auc = roc_auc_score(y_val, optimized_ensemble_proba)
opt_voting_f1 = f1_score(y_val, optimized_ensemble_pred)
opt_voting_custom_score = custom_score(y_val, optimized_ensemble_pred, optimized_ensemble_proba)

print(f"\nOptimized Weighted Voting Metrics:")
print(f"  Accuracy: {opt_voting_acc:.4f}")
print(f"  ROC-AUC: {opt_voting_auc:.4f}")
print(f"  F1-Score: {opt_voting_f1:.4f}")
print(f"  Custom Score: {opt_voting_custom_score:.4f}")

results['OptimizedVotingClassifier'] = {
    'val_auc': opt_voting_auc,
    'val_f1': opt_voting_f1,
    'predictions': optimized_ensemble_pred,
    'probabilities': optimized_ensemble_proba,
    'custom_score': opt_voting_custom_score,
    'optimized_weights': optimized_weights
}

# ============================================================================
# VOTING METHODS COMPARISON
# ============================================================================
print("\n" + "=" * 80)
print("VOTING METHODS COMPARISON")
print("=" * 80)

voting_methods = {
    'Standard Voting': results['VotingClassifier'],
    'Threshold-Aware Voting': results['ThresholdVotingClassifier'],
    'Optimized Voting': results['OptimizedVotingClassifier']
}

print(f"{'Voting Method':<25} | {'Accuracy':>10} | {'ROC-AUC':>10} | {'F1-Score':>10} | {'Custom Score':>12}")
print("-" * 85)

for name, res in voting_methods.items():
    acc = accuracy_score(y_val, res['predictions'])
    print(f"{name:<25} | {acc:>10.4f} | {res['val_auc']:>10.4f} | {res['val_f1']:>10.4f} | {res['custom_score']:>12.4f}")

# Find best voting method
best_voting = max(voting_methods.items(), key=lambda x: x[1]['custom_score'])
print(f"\nBest Voting Method: {best_voting[0]} (Custom Score: {best_voting[1]['custom_score']:.4f})")


--------------------------------------------------------------------------------
2. VOTING CLASSIFIER WITH THRESHOLD OPTIMIZATION
--------------------------------------------------------------------------------
Voting weights:
  XGBoost: 0.203
  XGBoost_rand: 0.200
  LightGBM_threshold: 0.200
  RandomForest: 0.199
  CatBoost_enhanced: 0.198

Training standard voting classifier...


/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/pyth

0:	total: 93.9ms	remaining: 3m 7s
100:	total: 13.9s	remaining: 4m 21s
200:	total: 18.4s	remaining: 2m 44s
300:	total: 22.8s	remaining: 2m 8s
400:	total: 27.2s	remaining: 1m 48s
500:	total: 31.6s	remaining: 1m 34s
600:	total: 36s	remaining: 1m 23s
700:	total: 40.3s	remaining: 1m 14s
800:	total: 44.7s	remaining: 1m 6s
900:	total: 49.1s	remaining: 59.9s
1000:	total: 53.5s	remaining: 53.4s
1100:	total: 57.9s	remaining: 47.3s
1200:	total: 1m 2s	remaining: 41.4s
1300:	total: 1m 6s	remaining: 35.8s
1400:	total: 1m 10s	remaining: 30.3s
1500:	total: 1m 15s	remaining: 25s
1600:	total: 1m 19s	remaining: 19.8s
1700:	total: 1m 23s	remaining: 14.7s
1800:	total: 1m 28s	remaining: 9.73s
1900:	total: 1m 32s	remaining: 4.81s
1999:	total: 1m 36s	remaining: 0us

Standard Voting Classifier Metrics:
  Accuracy: 0.7853
  ROC-AUC: 0.8441
  F1-Score: 0.6318
  Custom Score: 0.7393

--------------------------------------------------------------------------------
2b. THRESHOLD-AWARE VOTING CLASSIFIER
------------

In [ ]:
# ============================================================================
# ADVANCED ENSEMBLE STRATEGIES
# ============================================================================
print("\n" + "=" * 80)
print("ADVANCED ENSEMBLE STRATEGIES")
print("=" * 80)

# ============================================================================
# 3. DYNAMIC THRESHOLD ENSEMBLE
# ============================================================================
print("\n" + "-" * 80)
print("3. DYNAMIC THRESHOLD ENSEMBLE")
print("-" * 80)

class DynamicThresholdEnsemble:
    def __init__(self, models_info, confidence_threshold=0.1):
        self.models_info = models_info
        self.confidence_threshold = confidence_threshold
        self.model_thresholds = {}
        self.model_weights = {}
        
    def fit(self, X_val, y_val):
        """Determine dynamic thresholds based on prediction confidence"""
        total_weight = 0
        
        for name, info in self.models_info.items():
            # Get existing threshold or find optimal one
            if name == 'LightGBM_threshold' and hasattr(info.get('model'), 'threshold'):
                threshold = info['model'].threshold
            elif name == 'CatBoost_enhanced' and hasattr(info.get('model'), 'best_threshold'):
                threshold = info['model'].best_threshold
            else:
                probabilities = info['probabilities']
                threshold, _ = find_optimal_threshold(y_val, probabilities)
            
            self.model_thresholds[name] = threshold
            self.model_weights[name] = info['custom_score']
            total_weight += info['custom_score']
        
        # Normalize weights
        for name in self.model_weights:
            self.model_weights[name] /= total_weight
            
        print("Dynamic threshold configuration:")
        for name in self.model_thresholds:
            print(f"  {name}: threshold={self.model_thresholds[name]:.3f}, weight={self.model_weights[name]:.3f}")
    
    def predict(self, model_probabilities):
        """Make predictions using dynamic thresholding"""
        n_samples = len(next(iter(model_probabilities.values())))
        predictions = np.zeros(n_samples)
        
        for i in range(n_samples):
            # Calculate ensemble probability
            ensemble_prob = 0
            for name, prob_array in model_probabilities.items():
                ensemble_prob += self.model_weights[name] * prob_array[i]
            
            # Check confidence level
            confidence = abs(ensemble_prob - 0.5)
            
            if confidence > self.confidence_threshold:
                # High confidence: use ensemble probability with standard threshold
                predictions[i] = 1 if ensemble_prob > 0.5 else 0
            else:
                # Low confidence: use individual model thresholds with voting
                votes = 0
                total_weight = 0
                for name, prob_array in model_probabilities.items():
                    model_threshold = self.model_thresholds[name]
                    model_pred = 1 if prob_array[i] >= model_threshold else 0
                    weight = self.model_weights[name]
                    votes += weight * model_pred
                    total_weight += weight
                
                predictions[i] = 1 if votes / total_weight >= 0.5 else 0
        
        return predictions.astype(int)
    
    def predict_proba(self, model_probabilities):
        """Calculate ensemble probabilities"""
        ensemble_prob = np.zeros(len(next(iter(model_probabilities.values()))))
        for name, prob_array in model_probabilities.items():
            ensemble_prob += self.model_weights[name] * prob_array
        return ensemble_prob

# Prepare model probabilities for dynamic ensemble
dynamic_models = {
    'XGBoost': results['XGBoost'],
    'XGBoost_rand': results['XGBoost_rand'],
    'LightGBM_threshold': results['LightGBM_threshold'],
    'RandomForest': results['RandomForest'],
    'LightGBM': results['LightGBM'],
    'CatBoost_enhanced': results['CatBoost_enhanced']
}

model_probabilities = {name: info['probabilities'] for name, info in dynamic_models.items()}

# Create and fit dynamic ensemble
dynamic_ensemble = DynamicThresholdEnsemble(dynamic_models)
dynamic_ensemble.fit(X_val, y_val)

# Make predictions
dynamic_pred = dynamic_ensemble.predict(model_probabilities)
dynamic_proba = dynamic_ensemble.predict_proba(model_probabilities)

# Calculate metrics
dynamic_acc = accuracy_score(y_val, dynamic_pred)
dynamic_auc = roc_auc_score(y_val, dynamic_proba)
dynamic_f1 = f1_score(y_val, dynamic_pred)
dynamic_custom_score = custom_score(y_val, dynamic_pred, dynamic_proba)

print(f"\nDynamic Threshold Ensemble Metrics:")
print(f"  Accuracy: {dynamic_acc:.4f}")
print(f"  ROC-AUC: {dynamic_auc:.4f}")
print(f"  F1-Score: {dynamic_f1:.4f}")
print(f"  Custom Score: {dynamic_custom_score:.4f}")

results['DynamicEnsemble'] = {
    'val_auc': dynamic_auc,
    'val_f1': dynamic_f1,
    'predictions': dynamic_pred,
    'probabilities': dynamic_proba,
    'custom_score': dynamic_custom_score
}

# ============================================================================
# 4. BAYESIAN MODEL AVERAGING
# ============================================================================
print("\n" + "-" * 80)
print("4. BAYESIAN MODEL AVERAGING")
print("-" * 80)

def bayesian_model_averaging(model_probabilities, y_true, alpha=1.0):
    """Bayesian Model Averaging with Dirichlet prior"""
    n_models = len(model_probabilities)
    model_names = list(model_probabilities.keys())
    
    # Calculate likelihood for each model
    log_likelihoods = {}
    for name, proba in model_probabilities.items():
        # Avoid log(0) by clipping probabilities
        proba_clipped = np.clip(proba, 1e-10, 1 - 1e-10)
        log_likelihood = np.sum(y_true * np.log(proba_clipped) + 
                               (1 - y_true) * np.log(1 - proba_clipped))
        log_likelihoods[name] = log_likelihood
        print(f"  {name}: log-likelihood = {log_likelihood:.2f}")
    
    # Convert to weights using softmax (with Dirichlet prior)
    max_ll = max(log_likelihoods.values())
    exp_lls = {name: np.exp(ll - max_ll + alpha) for name, ll in log_likelihoods.items()}
    total_exp = sum(exp_lls.values())
    
    bayesian_weights = {name: exp_ll / total_exp for name, exp_ll in exp_lls.items()}
    
    print("Bayesian weights:")
    for name, weight in bayesian_weights.items():
        print(f"  {name}: {weight:.3f}")
    
    return bayesian_weights

# Calculate Bayesian weights
bayesian_weights = bayesian_model_averaging(model_probabilities, y_val)

# Create Bayesian ensemble
bayesian_proba = np.zeros_like(y_val, dtype=float)
for name, proba in model_probabilities.items():
    bayesian_proba += bayesian_weights[name] * proba

bayesian_pred = (bayesian_proba >= 0.5).astype(int)

# Calculate metrics
bayesian_acc = accuracy_score(y_val, bayesian_pred)
bayesian_auc = roc_auc_score(y_val, bayesian_proba)
bayesian_f1 = f1_score(y_val, bayesian_pred)
bayesian_custom_score = custom_score(y_val, bayesian_pred, bayesian_proba)

print(f"\nBayesian Model Averaging Metrics:")
print(f"  Accuracy: {bayesian_acc:.4f}")
print(f"  ROC-AUC: {bayesian_auc:.4f}")
print(f"  F1-Score: {bayesian_f1:.4f}")
print(f"  Custom Score: {bayesian_custom_score:.4f}")

results['BayesianEnsemble'] = {
    'val_auc': bayesian_auc,
    'val_f1': bayesian_f1,
    'predictions': bayesian_pred,
    'probabilities': bayesian_proba,
    'custom_score': bayesian_custom_score,
    'weights': bayesian_weights
}

# ============================================================================
# 5. RANK-BASED ENSEMBLE
# ============================================================================
print("\n" + "-" * 80)
print("5. RANK-BASED ENSEMBLE")
print("-" * 80)

def rank_based_ensemble(model_probabilities, weights=None):
    """Ensemble based on probability rankings"""
    if weights is None:
        weights = {name: 1/len(model_probabilities) for name in model_probabilities.keys()}
    
    n_samples = len(next(iter(model_probabilities.values())))
    
    # Convert probabilities to ranks
    model_ranks = {}
    for name, proba in model_probabilities.items():
        # Rank probabilities (higher probability = higher rank)
        ranks = np.argsort(np.argsort(proba)) / (len(proba) - 1)  # Normalize to [0,1]
        model_ranks[name] = ranks
    
    # Weighted average of ranks
    ensemble_ranks = np.zeros(n_samples)
    total_weight = sum(weights.values())
    
    for name, ranks in model_ranks.items():
        ensemble_ranks += (weights[name] / total_weight) * ranks
    
    return ensemble_ranks

# Use custom scores as weights for ranking
rank_weights = {name: info['custom_score'] for name, info in dynamic_models.items()}

# Create rank-based ensemble
rank_scores = rank_based_ensemble(model_probabilities, rank_weights)
rank_pred = (rank_scores >= 0.5).astype(int)

# Calculate metrics
rank_acc = accuracy_score(y_val, rank_pred)
rank_auc = roc_auc_score(y_val, rank_scores)
rank_f1 = f1_score(y_val, rank_pred)
rank_custom_score = custom_score(y_val, rank_pred, rank_scores)

print(f"\nRank-Based Ensemble Metrics:")
print(f"  Accuracy: {rank_acc:.4f}")
print(f"  ROC-AUC: {rank_auc:.4f}")
print(f"  F1-Score: {rank_f1:.4f}")
print(f"  Custom Score: {rank_custom_score:.4f}")

results['RankEnsemble'] = {
    'val_auc': rank_auc,
    'val_f1': rank_f1,
    'predictions': rank_pred,
    'probabilities': rank_scores,
    'custom_score': rank_custom_score
}

# ============================================================================
# COMPREHENSIVE ENSEMBLE COMPARISON
# ============================================================================
print("\n" + "=" * 80)
print("COMPREHENSIVE ENSEMBLE COMPARISON")
print("=" * 80)

all_ensembles = {
    'Standard Voting': results['VotingClassifier'],
    'Threshold-Aware Voting': results['ThresholdVotingClassifier'],
    'Optimized Voting': results['OptimizedVotingClassifier'],
    'Dynamic Threshold': results['DynamicEnsemble'],
    'Bayesian Averaging': results['BayesianEnsemble'],
    'Rank-Based': results['RankEnsemble']
}

print(f"{'Ensemble Method':<25} | {'Accuracy':>10} | {'ROC-AUC':>10} | {'F1-Score':>10} | {'Custom Score':>12}")
print("-" * 85)

# Sort by custom score
sorted_ensembles = sorted(all_ensembles.items(), key=lambda x: x[1]['custom_score'], reverse=True)

for name, res in sorted_ensembles:
    acc = accuracy_score(y_val, res['predictions'])
    print(f"{name:<25} | {acc:>10.4f} | {res['val_auc']:>10.4f} | {res['val_f1']:>10.4f} | {res['custom_score']:>12.4f}")

print(f"\nBest Overall Ensemble: {sorted_ensembles[0][0]} (Custom Score: {sorted_ensembles[0][1]['custom_score']:.4f})")

# ============================================================================
# ENSEMBLE OF ENSEMBLES (META-ENSEMBLE)
# ============================================================================
print("\n" + "-" * 80)
print("6. META-ENSEMBLE (ENSEMBLE OF ENSEMBLES)")
print("-" * 80)

# Select top 3 ensembles
top_ensembles = dict(sorted_ensembles[:3])
print(f"Using top 3 ensembles: {list(top_ensembles.keys())}")

# Create meta-ensemble
meta_weights = np.array([ens['custom_score'] for ens in top_ensembles.values()])
meta_weights = meta_weights / meta_weights.sum()

meta_proba = np.zeros_like(y_val, dtype=float)
for i, (name, ensemble) in enumerate(top_ensembles.items()):
    meta_proba += meta_weights[i] * ensemble['probabilities']
    print(f"  {name}: weight = {meta_weights[i]:.3f}")

meta_pred = (meta_proba >= 0.5).astype(int)

# Calculate metrics
meta_acc = accuracy_score(y_val, meta_pred)
meta_auc = roc_auc_score(y_val, meta_proba)
meta_f1 = f1_score(y_val, meta_pred)
meta_custom_score = custom_score(y_val, meta_pred, meta_proba)

print(f"\nMeta-Ensemble Metrics:")
print(f"  Accuracy: {meta_acc:.4f}")
print(f"  ROC-AUC: {meta_auc:.4f}")
print(f"  F1-Score: {meta_f1:.4f}")
print(f"  Custom Score: {meta_custom_score:.4f}")

results['MetaEnsemble'] = {
    'val_auc': meta_auc,
    'val_f1': meta_f1,
    'predictions': meta_pred,
    'probabilities': meta_proba,
    'custom_score': meta_custom_score
}

# Final comparison
if meta_custom_score > sorted_ensembles[0][1]['custom_score']:
    print(f"\n🎉 Meta-ensemble achieved better performance: {meta_custom_score:.4f} vs {sorted_ensembles[0][1]['custom_score']:.4f}")
else:
    print(f"\n📊 Best single ensemble remains: {sorted_ensembles[0][0]} with score {sorted_ensembles[0][1]['custom_score']:.4f}")

In [30]:


# STACKING CLASSIFIER
print("\n" + "-" * 80)
print("3. STACKING CLASSIFIER")
print("-" * 80)

from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

stacking_clf = StackingClassifier(
    estimators=voting_estimators,
    final_estimator=LogisticRegression(max_iter=1000, class_weight='balanced'),
    cv=5,
    n_jobs=-1,
    passthrough=False
)

# # Cross-validation for stacking classifier
# cv_scores_stacking = cross_val_score(stacking_clf, X_train, y_train, cv=skf, 
#                                   scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_stacking}")
# print(f"Mean CV ROC-AUC: {cv_scores_stacking.mean():.4f} (+/- {cv_scores_stacking.std() * 2:.4f})")

# Train stacking classifier
stacking_clf.fit(X_train, y_train)

y_pred_stacking = stacking_clf.predict(X_val)
y_proba_stacking = stacking_clf.predict_proba(X_val)[:, 1]

stacking_acc = accuracy_score(y_val, y_pred_stacking)
stacking_auc = roc_auc_score(y_val, y_proba_stacking)
stacking_f1 = f1_score(y_val, y_pred_stacking)
stacking_custom_score = custom_score(y_val, y_pred_stacking, y_proba_stacking)

print(f"\nstacking Classifier Metrics:")
print(f"  Accuracy: {stacking_acc:.4f}")
print(f"  ROC-AUC: {stacking_auc:.4f}")
print(f"  F1-Score: {stacking_f1:.4f}")
print(f"  Custom Score: {stacking_custom_score:.4f}")

results['stackingClassifier'] = {
    'model': stacking_clf,
    # 'cv_scores': cv_scores_stacking,
    'val_auc': stacking_auc,
    'val_f1': stacking_f1,
    'predictions': y_pred_stacking,
    'probabilities': y_proba_stacking,
    'custom_score': stacking_custom_score
}




--------------------------------------------------------------------------------
3. STACKING CLASSIFIER
--------------------------------------------------------------------------------


/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


0:	total: 217ms	remaining: 7m 14s
100:	total: 14.3s	remaining: 4m 28s
200:	total: 19s	remaining: 2m 49s
300:	total: 23.6s	remaining: 2m 13s
400:	total: 28.2s	remaining: 1m 52s
500:	total: 32.8s	remaining: 1m 38s
600:	total: 37.4s	remaining: 1m 27s
700:	total: 41.9s	remaining: 1m 17s
800:	total: 46.5s	remaining: 1m 9s
900:	total: 51.1s	remaining: 1m 2s
1000:	total: 55.7s	remaining: 55.6s
1100:	total: 1m	remaining: 49.2s
1200:	total: 1m 4s	remaining: 43.2s
1300:	total: 1m 9s	remaining: 37.4s
1400:	total: 1m 14s	remaining: 31.7s
1500:	total: 1m 18s	remaining: 26.2s
1600:	total: 1m 23s	remaining: 20.8s
1700:	total: 1m 27s	remaining: 15.4s
1800:	total: 1m 32s	remaining: 10.2s
1900:	total: 1m 36s	remaining: 5.03s
1999:	total: 1m 40s	remaining: 0us


/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/pyth

0:	total: 208ms	remaining: 6m 56s
0:	total: 256ms	remaining: 8m 31s
0:	total: 286ms	remaining: 9m 32s
0:	total: 280ms	remaining: 9m 20s
0:	total: 287ms	remaining: 9m 33s
100:	total: 44.7s	remaining: 14m 1s
100:	total: 45.8s	remaining: 14m 21s
100:	total: 46.2s	remaining: 14m 29s
100:	total: 46.3s	remaining: 14m 30s
100:	total: 46.3s	remaining: 14m 30s
200:	total: 1m 24s	remaining: 12m 39s
200:	total: 1m 25s	remaining: 12m 44s
200:	total: 1m 25s	remaining: 12m 44s
200:	total: 1m 25s	remaining: 12m 47s
200:	total: 1m 25s	remaining: 12m 48s
300:	total: 1m 52s	remaining: 10m 35s
300:	total: 1m 54s	remaining: 10m 45s
300:	total: 1m 54s	remaining: 10m 45s
300:	total: 1m 54s	remaining: 10m 46s
300:	total: 1m 54s	remaining: 10m 47s
400:	total: 2m 22s	remaining: 9m 29s
400:	total: 2m 24s	remaining: 9m 34s
400:	total: 2m 23s	remaining: 9m 34s
400:	total: 2m 24s	remaining: 9m 36s
400:	total: 2m 25s	remaining: 9m 40s
500:	total: 2m 51s	remaining: 8m 32s
500:	total: 2m 52s	remaining: 8m 34s
500:	to

In [48]:

# ============================================================================
# MODEL COMPARISON AND SELECTION
# ============================================================================
print("\n" + "=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)

# Create comparison dataframe
comparison_data = []
for model_name, model_results in results.items():
    if model_name in ['XGBoost', 'MetaEnsemble', 'DynamicEnsemble']:
        continue
    if 'cv_scores' in model_results:
        cv_mean = model_results['cv_scores'].mean()
        cv_std = model_results['cv_scores'].std()
    else:
        cv_mean = cv_std = np.nan
    
    comparison_data.append({
        'Model': model_name,
        'CV_AUC_Mean': cv_mean,
        'CV_AUC_Std': cv_std,
        'Val_AUC': model_results['val_auc'],
        'Val_F1': model_results['val_f1'],
        'Custom_Score': model_results['custom_score']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Custom_Score', ascending=False)

print("\nModel Performance Comparison:")
print("=" * 60)
for _, row in comparison_df.iterrows():
    # if pd.isna(row['CV_AUC_Mean']):
    #     print(f"{row['Model']:20s} | Val AUC: {row['Val_AUC']:.4f} | Val F1: {row['Val_F1']:.4f}")
    # else:
    #     print(f"{row['Model']:20s} | CV AUC: {row['CV_AUC_Mean']:.4f}±{row['CV_AUC_Std']:.4f} | "
    #           f"Val AUC: {row['Val_AUC']:.4f} | Val F1: {row['Val_F1']:.4f}")
    print(f"{row['Model']:20s} | Val AUC: {row['Val_AUC']:.4f} | Val F1: {row['Val_F1']:.4f} | "
          f"Custom Score: {row['Custom_Score']:.4f}")

# Select best model
best_model_name = comparison_df.iloc[0]['Model']
best_model_results = results[best_model_name]
# best_model_name = 'VotingClassifier'
# best_model_results = results['VotingClassifier']


print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   Validation AUC: {best_model_results['val_auc']:.4f}")
print(f"   Validation F1:  {best_model_results['val_f1']:.4f}")
print(f"   Custom Score:   {best_model_results['custom_score']:.4f}")

# ============================================================================
# FEATURE IMPORTANCE ANALYSIS
# ============================================================================
print("\n" + "=" * 80)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)

def get_feature_importance(model, feature_names, model_type):
    """Extract feature importance from different model types"""
    if hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_
    elif hasattr(model, 'coef_'):
        importance = abs(model.coef_[0])
    else:
        return None
    
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    return feature_importance

# Get feature importance from best tree-based models
models_for_importance = ['LightGBM', 'XGBoost', 'CatBoost', 'RandomForest']

print("Top 20 Most Important Features:")
print("-" * 50)

for model_name in models_for_importance:
    if model_name in results and 'model' in results[model_name]:
        importance_df = get_feature_importance(
            results[model_name]['model'], 
            X.columns, 
            model_name
        )
        
        if importance_df is not None:
            print(f"\n{model_name}:")
            top_features = importance_df.head(10)
            for idx, row in top_features.iterrows():
                original_name = feature_name_mapping.get(row['feature'], row['feature'])
                print(f"  {row['feature'][:30]:30s} | {row['importance']:.4f}")

# ============================================================================
# SAVE MODELS AND RESULTS
# ============================================================================
print("\n" + "=" * 80)
print("SAVING MODELS AND RESULTS")
print("=" * 80)

import pickle
import os

# Create models directory
os.makedirs('models', exist_ok=True)

# Save best model
best_model = best_model_results['model'] if 'model' in best_model_results else None
if best_model is not None:
    with open(f'models/best_model_{best_model_name.lower()}.pkl', 'wb') as f:
        pickle.dump(best_model, f)
    print(f"✅ Best model ({best_model_name}) saved to models/")

# Save feature columns and preprocessing objects
with open('models/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

with open('models/feature_name_mapping.pkl', 'wb') as f:
    pickle.dump(feature_name_mapping, f)

with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save results summary
results_summary = {
    'model_comparison': comparison_df.to_dict('records'),
    'best_model_name': best_model_name,
    'best_model_auc': best_model_results['val_auc'],
    'best_model_f1': best_model_results['val_f1'],
    'custom_score': best_model_results['custom_score'],
    'feature_count': len(feature_cols),
    'training_samples': len(X_train),
    'validation_samples': len(X_val)
}

with open('models/training_results.pkl', 'wb') as f:
    pickle.dump(results_summary, f)

print(f"✅ Feature preprocessing objects saved")
print(f"✅ Training results summary saved")

print("\n" + "=" * 80)
print("TRAINING COMPLETED SUCCESSFULLY!")
print("=" * 80)
print(f"🎯 Best Model: {best_model_name}")
print(f"📊 Validation AUC: {best_model_results['val_auc']:.4f}")
print(f"📈 Validation F1: {best_model_results['val_f1']:.4f}")
print(f"🧮 Total Features Used: {len(feature_cols)}")
print(f"💾 Models saved to 'models/' directory")
print("=" * 80)


MODEL COMPARISON SUMMARY

Model Performance Comparison:
ThresholdVotingClassifier | Val AUC: 0.8441 | Val F1: 0.6337 | Custom Score: 0.7419
ThresholdAwareEnsemble | Val AUC: 0.8430 | Val F1: 0.6275 | Custom Score: 0.7396
OptimizedVotingClassifier | Val AUC: 0.8441 | Val F1: 0.6322 | Custom Score: 0.7395
VotingClassifier     | Val AUC: 0.8441 | Val F1: 0.6318 | Custom Score: 0.7393
AdaptiveEnsemble     | Val AUC: 0.8430 | Val F1: 0.6299 | Custom Score: 0.7389
XGBoost_rand         | Val AUC: 0.8459 | Val F1: 0.6304 | Custom Score: 0.7367
LightGBM_threshold   | Val AUC: 0.8385 | Val F1: 0.6249 | Custom Score: 0.7364
StackedEnsemble      | Val AUC: 0.8439 | Val F1: 0.5712 | Custom Score: 0.7342
RandomForest         | Val AUC: 0.8347 | Val F1: 0.6199 | Custom Score: 0.7340
BayesianEnsemble     | Val AUC: 0.8389 | Val F1: 0.5520 | Custom Score: 0.7299
LightGBM             | Val AUC: 0.8389 | Val F1: 0.5520 | Custom Score: 0.7299
CatBoost_enhanced    | Val AUC: 0.8394 | Val F1: 0.6215 | Cust

In [50]:
# ============================================================================
# LOAD AND PREDICT TEST DATA
# ============================================================================

print("\n" + "=" * 80)
print("LOADING TEST DATA")
print("=" * 80)

df_test = pd.read_csv('./testB.csv')
print(f"Test data shape: {df_test.shape}")

# Store IDs
test_ids = df_test['id'].copy()

# ============================================================================
# APPLY SAME PREPROCESSING PIPELINE TO TEST DATA
# ============================================================================
print("\n" + "=" * 80)
print("PREPROCESSING TEST DATA")
print("=" * 80)

# Step 1: Apply the same imputation (using transform, not fit_transform)
print("Step 1: Applying imputation to test data...")
df_test_imputed = imputer.transform(df_test)

# Step 2: Apply the same feature engineering (using is_train=False)
print("Step 2: Creating features for test data...")
df_test_processed = engineer.create_all_features(
    df_test_imputed, 
    is_train=False, 
    target=None
)

# Step 3: Align features with training data
print("Step 3: Aligning test features with training data...")
# Use reindex to ensure we have the exact same features in the same order
X_test_full = df_test_processed.reindex(columns=feature_cols)

# Handle remaining missing values and infinities
X_test_full = X_test_full.replace([np.inf, -np.inf], 0)
X_test_full = X_test_full.fillna(0)

# Step 4: Select only the features that were selected during training
print("Step 4: Selecting trained features...")
X_test = X_test_full[selected_features].copy()

# Clean feature names to match training
X_test.columns = cleaned_feature_cols

# Check for feature alignment issues
missing_features = [col for col in selected_features if col not in df_test_processed.columns]
if missing_features:
    print(f"\nWarning: {len(missing_features)} features missing in test data (filled with zeros)")
    print(f"  Sample missing features: {missing_features[:5]}")
else:
    print("✓ All training features present in test data")

print(f"\nTest feature matrix shape: {X_test.shape}")
print(f"Training feature matrix shape: {X_train.shape}")

# Verify feature alignment
if X_test.shape[1] != X_train.shape[1]:
    print(f"\n⚠️  Warning: Feature count mismatch!")
    print(f"  Training: {X_train.shape[1]} features")
    print(f"  Test: {X_test.shape[1]} features")
else:
    print("✓ Feature counts match between train and test")

# Verify column names match
if list(X_test.columns) != list(X_train.columns):
    print("\n⚠️  Warning: Feature names don't match exactly")
    # Find differences
    train_only = set(X_train.columns) - set(X_test.columns)
    test_only = set(X_test.columns) - set(X_train.columns)
    if train_only:
        print(f"  Features only in training: {list(train_only)[:5]}")
    if test_only:
        print(f"  Features only in test: {list(test_only)[:5]}")
else:
    print("✓ Feature names match exactly between train and test")

# ============================================================================
# MAKE PREDICTIONS
# ============================================================================

print("\n" + "=" * 80)
print("MAKING PREDICTIONS")
print("=" * 80)

# Get predictions from best model
predictions = best_model.predict(X_test)
predictions_proba = best_model.predict_proba(X_test)[:, 1] if len(best_model.predict_proba(X_test).shape) >= 2 else best_model.predict_proba(X_test)

print(f"\nPredictions distribution:")
print(f"  Class 0: {(predictions == 0).sum():,} ({(predictions == 0).sum() / len(predictions) * 100:.1f}%)")
print(f"  Class 1: {(predictions == 1).sum():,} ({(predictions == 1).sum() / len(predictions) * 100:.1f}%)")

print(f"\nProbability statistics:")
print(f"  Mean probability: {predictions_proba.mean():.4f}")
print(f"  Median probability: {np.median(predictions_proba):.4f}")
print(f"  Std probability: {predictions_proba.std():.4f}")
print(f"  Min probability: {predictions_proba.min():.4f}")
print(f"  Max probability: {predictions_proba.max():.4f}")

# ============================================================================
# SAVE RESULTS
# ============================================================================

print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

# Create submission dataframe
submission = pd.DataFrame({
    'id': test_ids,
    'label': predictions
})

# Save to CSV
output_path = './submit.csv'
submission.to_csv(output_path, index=False)

print(f"✅ Submission saved to: {output_path}")
print(f"   Submission shape: {submission.shape}")
print(f"\nFirst 10 predictions:")
print(submission.head(10))

# Also save with probabilities for analysis
submission_with_proba = pd.DataFrame({
    'id': test_ids,
    'label': predictions,
    'probability': predictions_proba
})

output_path_proba = './submit_with_probabilities.csv'
submission_with_proba.to_csv(output_path_proba, index=False)
print(f"\n✅ Predictions with probabilities saved to: {output_path_proba}")

# ============================================================================
# SAVE PREPROCESSING OBJECTS FOR LATER USE
# ============================================================================
print("\n" + "=" * 80)
print("SAVING PREPROCESSING PIPELINE")
print("=" * 80)

# Save the imputer and engineer for future use
with open('models/imputer.pkl', 'wb') as f:
    pickle.dump(imputer, f)

with open('models/engineer.pkl', 'wb') as f:
    pickle.dump(engineer, f)

with open('models/selected_features.pkl', 'wb') as f:
    pickle.dump(selected_features, f)

print("✅ Preprocessing pipeline saved:")
print("   - models/imputer.pkl")
print("   - models/engineer.pkl")
print("   - models/selected_features.pkl")

print("\n" + "=" * 80)
print("PIPELINE COMPLETE!")
print("=" * 80)
print(f"🎯 Best Model: {best_model_name}")
print(f"📊 Validation AUC: {best_model_results['val_auc']:.4f}")
print(f"📈 Validation F1: {best_model_results['val_f1']:.4f}")
print(f"🧮 Features Used: {X_test.shape[1]}")
print(f"🔮 Predictions Made: {len(predictions):,}")
print(f"💾 Files saved: submit.csv, submit_with_probabilities.csv")
print("=" * 80)


LOADING TEST DATA
Test data shape: (19999, 87)

PREPROCESSING TEST DATA
Step 1: Applying imputation to test data...
Step 2: Creating features for test data...

ADVANCED FEATURE ENGINEERING
Creating usage pattern features...
Creating temporal features...
Creating behavioral segments...
Creating revenue features...
Creating engagement metrics...
Creating risk indicators...
Creating interaction features...
Creating statistical features...
Creating target-encoded features...
Creating polynomial features...
Creating aggregation features...
Applying mathematical transformations...

✓ Feature engineering complete. Total features: 351
Step 3: Aligning test features with training data...
Step 4: Selecting trained features...

  Sample missing features: ['is_data_centric_log1p', 'revenue_per_gb_log1p']

Test feature matrix shape: (19999, 120)
Training feature matrix shape: (50918, 120)
✓ Feature counts match between train and test
✓ Feature names match exactly between train and test

MAKING PRE